<div style="font-size: 15.5px; line-height: 1.6; background: #ffffff; padding: 20px 28px; border-radius: 8px; color: #1a365d;">

<h2 style="font-size: 24px; margin: 4px 0 16px 0; color: #1a365d; border-bottom: 2px solid #3182ce; padding-bottom: 8px;">Module 11 · Notebook 06 — LangGraph</h2>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">The Problem LangGraph Solves</h3>

<p style="margin: 10px 0; color: #1a365d;">In the last three notebooks, we hit real limitations:</p>

<table style="font-size: 15px; margin: 10px 0; border-collapse: collapse; width: 100%; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.06); color: #1a365d;">
  <thead>
    <tr style="background: #ebf4ff;">
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Notebook</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Problem</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">03 (ReAct)</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Non-deterministic — same question → different answers. Loops on repeated tool calls.</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">04 (Planning)</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Dependencies between steps are fragile — placeholder substitution broke.</td></tr>
    <tr><td style="padding: 10px 14px; color: #1a365d;">05 (Multi-agent)</td><td style="padding: 10px 14px; color: #1a365d;">Agent-to-agent coordination is hard to build by hand.</td></tr>
  </tbody>
</table>

<p style="margin: 10px 0; color: #1a365d;"><strong>LangGraph solves all three</strong> by replacing "let the LLM decide everything" with <strong>explicit state machines</strong>.</p>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">The Core Idea: State Machines</h3>

<p style="margin: 10px 0; color: #1a365d;">A <strong>state machine</strong> is a graph where:</p>

<ul style="margin: 10px 0; padding-left: 26px; color: #1a365d;">
  <li><strong>Nodes</strong> = functions that do work (call a tool, generate an answer, etc.)</li>
  <li><strong>Edges</strong> = connections that say "after this node, go to that node"</li>
  <li><strong>State</strong> = shared data that flows through all nodes</li>
</ul>

<p style="margin: 10px 0; color: #1a365d;">The flow is <strong>explicit</strong> — the graph defines it, not the LLM.</p>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">A Visual Example</h3>

<pre style="font-size: 14px; line-height: 1.5; margin: 10px 0; padding: 14px 18px; background: #ebf4ff; color: #1a365d; border-radius: 6px; border-left: 4px solid #3182ce; overflow-x: auto;"><code>        ┌──────────┐
        │  START   │
        └────┬─────┘
             ▼
        ┌──────────┐
        │  Agent   │◄────────┐
        │ (decide) │         │
        └────┬─────┘         │
             ▼               │
        ┌──────────────┐     │
        │ Call tool?   │─────┘   (loop back if more tools needed)
        └──────┬───────┘
             │ no more tools
             ▼
        ┌──────────┐
        │  END     │
        └──────────┘</code></pre>

<p style="margin: 10px 0; color: #1a365d;">Unlike ReAct, the <strong>loop is bounded</strong> and <strong>explicit</strong>. The graph knows it has two states: "deciding" and "done".</p>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">Why This Fixes Notebooks 03 and 04</h3>

<table style="font-size: 15px; margin: 10px 0; border-collapse: collapse; width: 100%; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.06); color: #1a365d;">
  <thead>
    <tr style="background: #ebf4ff;">
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Problem</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">How LangGraph fixes it</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Loops on repeated tool calls</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">You define the exact edges — no ambiguity, no loops unless you draw one</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Placeholder substitution</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">State carries real values from node to node — no placeholders needed</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Non-deterministic flow</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">The graph is the same every run — only the LLM's decisions inside nodes vary</td></tr>
    <tr><td style="padding: 10px 14px; color: #1a365d;">Multi-agent coordination</td><td style="padding: 10px 14px; color: #1a365d;">Each agent is just a node — they share state automatically</td></tr>
  </tbody>
</table>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">The Three Pieces of a LangGraph Agent</h3>

<table style="font-size: 15px; margin: 10px 0; border-collapse: collapse; width: 100%; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.06); color: #1a365d;">
  <thead>
    <tr style="background: #ebf4ff;">
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Piece</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">What it is</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>State</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">A typed dictionary that holds all shared data — question, tool results, final answer</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>Nodes</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Functions that take state and return updated state</td></tr>
    <tr><td style="padding: 10px 14px; color: #1a365d;"><strong>Edges</strong></td><td style="padding: 10px 14px; color: #1a365d;">Rules for which node runs next (fixed or conditional)</td></tr>
  </tbody>
</table>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">What We'll Build</h3>

<table style="font-size: 15px; margin: 10px 0; border-collapse: collapse; width: 100%; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.06); color: #1a365d;">
  <thead>
    <tr style="background: #ebf4ff;">
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Cell</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">What it does</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>1</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Install LangGraph and imports</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>2</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Define the state schema</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>3</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Define tools (reuse from Notebook 04)</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>4</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Write the agent node — decides what to do</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>5</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Write the tool node — runs the chosen tool</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>6</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Wire the graph — nodes + edges + conditional routing</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>7</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Compile and run</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>8</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Visualise the graph (real image!)</td></tr>
    <tr><td style="padding: 10px 14px; color: #1a365d;"><strong>9</strong></td><td style="padding: 10px 14px; color: #1a365d;">Compare with ReAct / Planning — same question, three patterns</td></tr>
  </tbody>
</table>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">Why This Matters for AgriVoice</h3>

<p style="margin: 10px 0; color: #1a365d;">AgriVoice's workflow is a <strong>state machine</strong>:</p>

<pre style="font-size: 14px; line-height: 1.5; margin: 10px 0; padding: 14px 18px; background: #ebf4ff; color: #1a365d; border-radius: 6px; border-left: 4px solid #3182ce; overflow-x: auto;"><code>[START] → [Receive Photo] → [Diagnose] → [Search Treatment]
       → [Translate] → [Speak Answer] → [END]</code></pre>

<p style="margin: 10px 0; color: #1a365d;">Some steps are <strong>conditional</strong>:</p>

<ul style="margin: 10px 0; padding-left: 26px; color: #1a365d;">
  <li>If the user asked in Yoruba → route to translate</li>
  <li>If the user asked in English → skip translate</li>
  <li>If the confidence is low → route to "ask for another photo"</li>
</ul>

<p style="margin: 10px 0; color: #1a365d;"><strong>LangGraph is how you build that.</strong> Every production voice/vision agent uses this pattern.</p>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">Key Terms</h3>

<table style="font-size: 15px; margin: 10px 0; border-collapse: collapse; width: 100%; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.06); color: #1a365d;">
  <thead>
    <tr style="background: #ebf4ff;">
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Term</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>StateGraph</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">The main LangGraph class — you build your graph with it</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>Node</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">A function that reads state and returns an update</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>Edge</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">A connection between two nodes</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>Conditional edge</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">An edge that reads state and decides where to go next</td></tr>
    <tr><td style="padding: 10px 14px; color: #1a365d;"><strong>Compile</strong></td><td style="padding: 10px 14px; color: #1a365d;">Turn the graph definition into a runnable app</td></tr>
  </tbody>
</table>

</div>